# ทดลอง PDF → Text สำหรับ PDPA

รันทีละเซลล์และเทียบกับภาพ PDF ก่อนตัดสินใจเลือกวิธี extraction หรือ cleaning

Notebook นี้เป็นพื้นที่ทดลอง ไม่เปลี่ยนสคริปต์จริง และไม่บันทึกข้อความจนกว่าจะเปิด `SAVE_OUTPUT`

**เริ่มใช้งาน:** เลือก Kernel → Python Environments → `Backend/.venv/Scripts/python.exe`

## 1. ตั้งค่าหน้าและภาษา

เริ่มภาษาไทยหน้า 1 จากนั้นลองหน้าอื่นและภาษาอังกฤษ เลขด้านล่างคือลำดับหน้าใน PDF เริ่มจาก 1 ไม่ใช่เลขหน้าที่พิมพ์บนกระดาษ

In [ ]:
from pathlib import Path
import re
import sys

import pymupdf
from IPython.display import Image, display

# เปลี่ยนสองค่านี้ แล้วรันเซลล์ด้านล่างใหม่
LANGUAGE = "th"  # "th" หรือ "en"
PDF_PAGE = 1
SAVE_OUTPUT = False  # เปิดเมื่อคุณตรวจแล้วและต้องการเก็บตัวอย่าง

# รองรับการเปิดจาก rag, โฟลเดอร์โปรเจกต์, Backend หรือ notebooks
project_dir = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    for candidate in (folder, folder / "Bilingual-PDPA-RAG-Engine"):
        if (candidate / "Data" / "raw" / "PDPA (TH).pdf").is_file():
            project_dir = candidate
            break
    if project_dir is not None:
        break

if project_dir is None:
    raise FileNotFoundError("เปิด Notebook จากโฟลเดอร์โปรเจกต์นี้ก่อน")
if LANGUAGE not in ("th", "en"):
    raise ValueError("LANGUAGE ต้องเป็น th หรือ en")

source_file = "PDPA (TH).pdf" if LANGUAGE == "th" else "PDPA (Eng).pdf"
pdf_path = project_dir / "Data" / "raw" / source_file
print("Python:", sys.executable)
print("PyMuPDF:", pymupdf.__version__)
print("PDF:", pdf_path)

## 2. ดูภาพต้นฉบับและข้อความหน้าเดียวกัน

ตรวจสระ วรรณยุกต์ เลขมาตรา ลำดับข้อความ และส่วนที่ตกหล่น จำนวนตัวอักษรไม่ได้บอกว่าข้อความถูกต้อง

In [ ]:
with pymupdf.open(pdf_path) as document:
    if not 1 <= PDF_PAGE <= len(document):
        raise ValueError(f"PDF_PAGE ต้องอยู่ระหว่าง 1 ถึง {len(document)}")
    page = document[PDF_PAGE - 1]
    page_image = page.get_pixmap(matrix=pymupdf.Matrix(1.5, 1.5)).tobytes("png")
    raw_text = page.get_text("text", sort=True)
    unsorted_text = page.get_text("text", sort=False)
    total_pages = len(document)

print(f"PDF page {PDF_PAGE}/{total_pages}")
display(Image(data=page_image, width=750))
print("--- ข้อความดิบ: sort=True ---")
print(raw_text)

## 3. เปรียบเทียบลำดับข้อความ

ลองเทียบกับภาพว่า `sort=True` และ `sort=False` แบบไหนอ่านถูกกว่า อย่าเลือกจากความสวยงามเพียงอย่างเดียว

In [ ]:
print("--- sort=False ---")
print(unsorted_text)
print("ผลสองแบบเหมือนกัน:", raw_text == unsorted_text)
print("จำนวนตัวอักษรดิบ:", len(raw_text))
print("จำนวนตัวอักษร ำ:", raw_text.count("ำ"))
print("จำนวนรูปแบบช่องว่าง + า:", raw_text.count(" า"))

## 4. หาตำแหน่งที่ควรตรวจ

รายการนี้เป็นเพียงรูปแบบที่น่าสงสัย ไม่ใช่รายการคำผิดที่ยืนยันแล้ว และไม่ได้ครอบคลุมข้อผิดพลาดทั้งหมด

การทำ Unicode normalization อย่างเดียวไม่สามารถเดาว่าสัญลักษณ์ที่หายไปควรเป็นอะไรได้

In [ ]:
if LANGUAGE == "th":
    matches = list(re.finditer(r"[ก-ฮ][่้๊๋]? +า", raw_text))
    print("ตำแหน่งที่ควรตรวจ:", len(matches))
    for match in matches[:30]:
        start = max(0, match.start() - 25)
        end = min(len(raw_text), match.end() + 35)
        print(repr(raw_text[start:end]))
else:
    print("การตรวจรูปแบบนี้ใช้เฉพาะภาษาไทย")

## 5. ทดลองแก้เฉพาะข้อความที่ตรวจแล้ว

กฎด้านล่างเป็นตัวอย่างจากคำที่เจ้าของรายงาน ให้เทียบต้นฉบับทุกตำแหน่งก่อนยอมรับกฎ

อย่าแทน ` า` → `ำ` ทั่วเอกสาร เพราะอาจแก้ช่องว่างจริงผิด เก็บ `raw_text` ไว้เสมอ

In [ ]:
# เพิ่มกฎทีละรายการ หลังตรวจตำแหน่งที่พบกับ PDF แล้ว
corrections = {"จ ากัด": "จำกัด"} if LANGUAGE == "th" else {}

cleaned_text = raw_text
for before, after in corrections.items():
    count = cleaned_text.count(before)
    print(f"{before!r} → {after!r}: พบ {count} ตำแหน่ง")
    cleaned_text = cleaned_text.replace(before, after)

print("--- ตัวอย่างหลังแก้ ยังต้องตรวจ ---")
print(cleaned_text)

## 6. บันทึกตัวอย่าง (เลือกทำ)

กลับไปตั้ง `SAVE_OUTPUT = True` แล้วรันใหม่เมื่ออยากบันทึก เก็บข้อความดิบกับข้อความหลังแก้แยกกัน พร้อมชื่อไฟล์และลำดับหน้า ไม่เขียนทับไฟล์ที่มีอยู่

`cleaned` หมายถึงผ่านกฎตัวอย่าง ไม่ได้หมายความว่าตรวจความถูกต้องครบแล้ว

In [ ]:
if SAVE_OUTPUT:
    output_dir = project_dir / "Data" / "extracted"
    stem = f"pdpa_{LANGUAGE}_notebook_page_{PDF_PAGE:03d}"
    raw_path = output_dir / f"{stem}_raw.txt"
    cleaned_path = output_dir / f"{stem}_cleaned.txt"
    existing = [path for path in (raw_path, cleaned_path) if path.exists()]
    if existing:
        raise FileExistsError(f"มีผลลัพธ์แล้ว: {existing}; ย้ายไฟล์เดิมก่อน")
    header = f"Source: {source_file}\nLanguage: {LANGUAGE}\nPDF page: {PDF_PAGE}\n\n"
    output_dir.mkdir(parents=True, exist_ok=True)
    raw_path.write_text(header + raw_text, encoding="utf-8")
    cleaned_path.write_text(header + cleaned_text, encoding="utf-8")
    print("บันทึก:", raw_path)
    print("บันทึก:", cleaned_path)
else:
    print("Preview only: ยังไม่ได้บันทึกข้อความลงไฟล์")

## 7. บันทึกสิ่งที่คุณพบ

แก้ Markdown เซลล์นี้หลังทดลอง เพื่อให้แยกผลจริงออกจากข้อสันนิษฐาน

- ภาษา / ลำดับหน้า PDF ที่ตรวจ: ยังไม่ได้กรอก
- วิธี extraction / sort ที่เลือก: ยังไม่ได้ตัดสินใจ
- ข้อความผิด → ข้อความตามต้นฉบับ: ยังไม่ได้กรอก
- กฎแก้ข้อความและตำแหน่งที่ตรวจครบแล้ว: ยังไม่ได้กรอก
- ปัญหาที่เหลือ: ยังไม่ได้กรอก
- ลองอย่างน้อยช่วงต้น กลาง และท้ายของทั้งสองภาษา ก่อนแก้ pipeline จริง

เมื่อรูปแบบที่เลือกผ่านการตรวจแล้ว จึงนำกฎที่ยืนยันแล้วไปใช้ใน `src/backend/pipelines/` และเตรียม JSONL ต่อ